In [53]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

In [54]:
class ThreeState():
    def __init__(self, E0=0, E1=-np.log(2), E2=0, kT=1/3):
        self.E0 = E0
        self.E1 = E1
        self.E2 = E2
        self.kT = kT
        self.energies = [self.E0, self.E1, self.E2]

    def partitionFunction(self):
        Q = 0
        for E in self.energies:
            Q += np.exp(-(E / self.kT))
        return Q

    def normProbabilityDensity(self, state):
        Pi = 1 / self.partitionFunction() * np.exp(-self.energies[state] / self.kT)
        return Pi

    def thermalAverage(self):
        energies = np.array(self.energies)
        return np.sum(energies * np.exp(-energies / self.kT)) / self.partitionFunction()

    def metropolisSample(self, steps=100000, initState=0):
        currentState = initState
        stateCounts = np.zeros(len(self.energies))
        for _ in range(steps):
            propState = np.random.choice([0, 1, 2])
            dE = self.energies[propState] - self.energies[currentState]
            acceptProp = min(1, np.exp(-dE / self.kT))
            if np.random.rand() < acceptProp:
                currentState = propState

            stateCounts[currentState] += 1

        P_estimated = stateCounts / steps
        E_avg_estimated = np.sum(P_estimated * np.array(self.energies))

        return P_estimated, E_avg_estimated

    def metropolisVaryingSample(self, steps=10000, initState=0):
        T = np.array([
        [0.5, 0.5, 0.0],
        [0.5, 0.0, 0.5],
        [0.0, 0.5, 0.5]
        ])

        currentState = initState
        stateCounts = np.zeros(len(self.energies))
        states = [0,1,2]

        for _ in range(steps):
            propState = np.random.choice(states, p=T[currentState])
            dE = self.energies[propState] - self.energies[currentState]
            acceptProp = min(1, np.exp(-dE / self.kT))
            if np.random.rand() < acceptProp:
                currentState = propState

            stateCounts[currentState] += 1

        P_estimated = stateCounts / steps
        E_avg_estimated = np.sum(P_estimated * np.array(self.energies))

        return P_estimated, E_avg_estimated

In [55]:
ts = ThreeState()

print(f"Q={ts.partitionFunction()}")
for i in range(3):
    print(f"P{i} = {ts.normProbabilityDensity(i)}")

print(f"E_avg = {ts.thermalAverage()}")

Q=10.000000000000002
P0 = 0.09999999999999998
P1 = 0.8
P2 = 0.09999999999999998
E_avg = -0.5545177444479562


In [56]:
ts.metropolisSample()

(array([0.1003 , 0.79926, 0.10044]), np.float64(-0.5540048155343419))

In [57]:
ts.metropolisVaryingSample()

(array([0.0988, 0.8079, 0.0933]), np.float64(-0.5599936071743797))

In [58]:
class Node:
    def __init__ (self , index , energy):
        self.index = index
        self.energy = energy

class System:
    def __init__ (self, energies, kT, Tij):
        self.energies = energies
        self.kT = kT
        self.Tij = Tij
        self.nodes = [Node(i,e) for i,e in enumerate (self.energies )]
        self.state = self.nodes [0]
        self.history = []

    def proposed_state (self):
        j = np.random.choice(len(self.nodes), p=self.Tij[self.state.index])
        return self.nodes[j]
    
    def step(self):
        proposed_state = self.proposed_state()
        i = self.state.index
        j = proposed_state.index
        delta_e = proposed_state.energy - self.state.energy
        proposal_ratio = self.Tij[j, i] / self.Tij[i, j]
        acc_prob = min(1.0, proposal_ratio * np.exp(-delta_e / self.kT))

        if np.random.rand() < acc_prob:
            self.state = proposed_state

        self.history.append(self.state.index)

    def run_Metropolis_Monte_Carlo (self, n_steps):
        for _ in range( n_steps ):
            self.step()

In [59]:
energies = [0, -np.log(2), 0]
kT = 1/3

T_asymmetric = np.array([
    [0.1, 0.8, 0.1], 
    [0.2, 0.5, 0.3], 
    [0.4, 0.4, 0.2]   
])

sys = System(energies, kT, T_asymmetric)
sys.run_Metropolis_Monte_Carlo(100000)

counts = np.bincount(sys.history, minlength=len(energies))
P_sampled = counts / len(sys.history)

E_avg_sampled = np.sum(P_sampled * np.array(energies))

print("Sampled P0, P1, P2:", np.round(P_sampled, 4))
print("Sampled E_avg:", round(E_avg_sampled, 6))

Sampled P0, P1, P2: [0.1 0.8 0.1]
Sampled E_avg: -0.554532
